In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [19]:
column_names = ['target', 'id', 'date', 'flag', 'user', 'text']

In [23]:
df = pd.read_csv("sentiment.csv", encoding='ISO-8859-1')

In [24]:
df.head()

,target,id,date,flag,user,text
0,1,2064229792,Sun Jun 07 05:38:57 PDT 2009,NO_QUERY,PinoyTarsier,@indykitty *hug indykitty* sleep tight indy...
1,0,2063435330,Sun Jun 07 02:38:52 PDT 2009,NO_QUERY,Hannah_oxberry,@Shough yeah I feel really bad for them tryin...
2,0,2251087443,Sat Jun 20 02:23:50 PDT 2009,NO_QUERY,meabhaline,@embeep sorry about your sadness I'll be home...
3,0,2066987792,Sun Jun 07 11:32:31 PDT 2009,NO_QUERY,PDKG,Couldn't spend time with the family cuz of wor...
4,1,1557513350,Sun Apr 19 04:34:04 PDT 2009,NO_QUERY,eulaivi,is new on twitter


In [25]:
df.shape

(50000, 6)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   target  50000 non-null  int64 
 1   id      50000 non-null  int64 
 2   date    50000 non-null  object
 3   flag    50000 non-null  object
 4   user    50000 non-null  object
 5   text    50000 non-null  object
dtypes: int64(2), object(4)
memory usage: 2.3+ MB


In [27]:
np.unique(df['flag'])  #If all values are NO_QUERY , then why there is need to keep this column

array(['NO_QUERY'], dtype=object)

In [28]:
#Id of tweet will not affect the result. We can remove it
#There are no missing values in data
#Date , user and text all are of type string: {can convert date to int, but what about user and text} 

#But here target is onlyy affected by the text, so while training the model, we do not need to consider other columns


In [29]:
#The given dataset has only positive or negative tweets, we can make target=0 for negative , 1 for positive



In [30]:
df['target'].value_counts()

target
1    25000
0    25000
Name: count, dtype: int64

In [31]:
#stopwords(I, my, he, she..) do not affect the sentiment of data, so we can extract them from the text and can remove them 

import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [32]:
stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [33]:
#Stemming: Procedure to convert words which are of same category to single word(root word)(like actor, acting, actress --> act)

port_stem = PorterStemmer()

def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)  #removing all characters from the contant except alphabets
    stemmed_content = stemmed_content.lower() # converting content to lower case
    stemmed_content = stemmed_content.split()

    stemmed_content = [port_stem.stem(word) for word in stemmed_content if word not in stopwords.words('english')] #steemming all words other than stopwords
    stemmed_content = ' '.join(stemmed_content)

    return stemmed_content
    

In [34]:
#Complete datset has 16 million rows, it will take around 50 minutes for this coversion. So, for now we are taking only 50,000
# rows randomly from the complete dataset and will train our model on that dataset

In [35]:
df['stemmed_content'] = df['text'].apply(stemming)

In [36]:
df['stemmed_content'].head()

0             indykitti hug indykitti sleep tight indi
1    shough yeah feel realli bad tri best help know...
2    embeep sorri sad home next week faff stalk cel...
3        spend time famili cuz work went beach without
4                                          new twitter
Name: stemmed_content, dtype: object

In [37]:
X = df['stemmed_content'].values
y = df['target'].values

In [38]:
X

array(['indykitti hug indykitti sleep tight indi',
       'shough yeah feel realli bad tri best help know say',
       'embeep sorri sad home next week faff stalk celeb twitter', ...,
       'strangegod say love friday strip much may steal wallet whenev meet',
       'got back church broke foot caught holyghost',
       'infam got point wre want break control search hour freak blast chard angri'],
      dtype=object)

In [39]:
# train-test split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=2)



In [40]:
X_train

array(['studi like weekend think million thing would rather',
       'wait get new laptop http bit ly cvlno',
       'say yey plurk phone http plurk com p z rtn', ...,
       'citytravelbug green',
       'karasalazar sorri watch movi bili ray cyru',
       'ecosalon hmmm grrr sinc affect blog stat'], dtype=object)

In [41]:
# now convert text of each row to a numerical value using tfidfVectorizer

tf_vector =TfidfVectorizer()

X_train_tf = tf_vector.fit_transform(X_train)
X_test_tf = tf_vector.transform(X_test)

In [42]:
classifier = LogisticRegression()

classifier.fit(X_train_tf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [43]:
y_pred_train = classifier.predict(X_train_tf)
training_acc = accuracy_score(y_pred_train, y_train)

In [44]:
training_acc

0.834475

In [45]:
y_pred_test = classifier.predict(X_test_tf)
testing_acc = accuracy_score(y_pred_test, y_test)

In [46]:
testing_acc

0.7468

In [47]:
import pickle

In [48]:
#saving the trained model
filename = 'trained_model.sav'

pickle.dump(classifier, open(filename, 'wb'))

In [49]:
#using saved model from the file

model = pickle.load(open(filename, 'rb'))

In [50]:
model

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'
